# Does age affect healthy and stroke gait differently?

This analysis separates three questions:
1. Do gait features change with age within healthy participants?
2. Do they change with a different slope within stroke participants?
3. Does the trained stroke model perform differently in age ranges where healthy and stroke participants overlap?

The interaction term `age × stroke` is the direct test of whether age acts differently in the two groups. The analysis is participant-level: trials are averaged within participant before fitting the interaction models.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from features import voisard
PROCESSED = PROJECT_ROOT / 'data' / 'processed'; INTERIM = PROJECT_ROOT / 'data' / 'interim'
FEATURES = ['cadence_steps_per_min', 'stride_time_mean_s', 'stride_time_cv_mean', 'lb_accel_rms', 'foot_accel_rms_mean', 'he_accel_rms']
manifest = pd.read_csv(INTERIM / 'ml_readiness_manifest.csv')
age_map = manifest[manifest.dataset_id.eq('voisard_2025')].groupby('subject').age.first().astype(float).to_dict()
trial_features = voisard.build_feature_table()
participant = trial_features.groupby(['subject', 'label'], as_index=False)[FEATURES].mean()
participant['age'] = participant.subject.map(age_map); participant['stroke'] = participant.label.eq('CVA').astype(int)
participant['age_group'] = pd.cut(participant.age, bins=[17, 39, 59, 100], labels=['young_18_39', 'middle_40_59', 'older_60_plus'])
participant = participant.dropna(subset=['age']).reset_index(drop=True)
print('Participants:', len(participant)); print(participant.groupby(['age_group','label']).size()); print(participant.groupby('label').age.agg(['mean','std','min','max']))

Participants: 122
age_group      label
young_18_39    CVA       0
               HS       43
middle_40_59   CVA      27
               HS       15
older_60_plus  CVA      22
               HS       15
dtype: int64
            mean        std   min   max
label                                  
CVA    58.979592   9.465925  41.0  83.0
HS     39.753425  20.183038  18.0  87.0


C:\Users\frank\AppData\Local\Temp\ipykernel_28300\139861529.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print('Participants:', len(participant)); print(participant.groupby(['age_group','label']).size()); print(participant.groupby('label').age.agg(['mean','std','min','max']))


In [2]:
def interaction_fit(frame, feature):
    sub = frame[['age', 'stroke', feature]].dropna().copy(); age10 = (sub.age - sub.age.mean()) / 10.0
    y = sub[feature].to_numpy(float); g = sub.stroke.to_numpy(float); X = np.column_stack([np.ones(len(sub)), age10, g, age10 * g])
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None); resid = y - X @ beta; df = len(y) - X.shape[1]
    sigma2 = (resid @ resid) / df; cov = sigma2 * np.linalg.inv(X.T @ X); se = np.sqrt(np.diag(cov)); tvals = beta / se; pvals = 2 * stats.t.sf(np.abs(tvals), df)
    slope_stroke = beta[1] + beta[3]; slope_var = cov[1,1] + cov[3,3] + 2 * cov[1,3]; slope_se = np.sqrt(slope_var); slope_t = slope_stroke / slope_se; slope_p = 2 * stats.t.sf(abs(slope_t), df)
    return {'feature': feature, 'n': len(sub), 'healthy_age_slope_per_10y': beta[1], 'stroke_age_slope_per_10y': slope_stroke, 'age_by_stroke_interaction': beta[3], 'interaction_p': pvals[3], 'age_adjusted_stroke_difference': beta[2], 'age_main_p': pvals[1], 'stroke_main_p': pvals[2], 'stroke_slope_p': slope_p}

interaction_rows = [interaction_fit(participant, feature) for feature in FEATURES]
interactions = pd.DataFrame(interaction_rows).sort_values('interaction_p').reset_index(drop=True)
p = interactions['interaction_p'].to_numpy(float); order = np.argsort(p); ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
q = np.empty_like(ranked); q[::-1] = np.minimum.accumulate(ranked[::-1]); q = np.clip(q, 0, 1); adjusted = np.empty_like(q); adjusted[order] = q
interactions['interaction_q_bh'] = adjusted
print(interactions.round(4).to_string(index=False))

              feature   n  healthy_age_slope_per_10y  stroke_age_slope_per_10y  age_by_stroke_interaction  interaction_p  age_adjusted_stroke_difference  age_main_p  stroke_main_p  stroke_slope_p  interaction_q_bh
         he_accel_rms 122                    -0.0371                    0.0044                     0.0416         0.0391                         -0.1270      0.0000         0.0001          0.8128            0.1483
         lb_accel_rms 122                    -0.0733                   -0.0217                     0.0516         0.0494                         -0.2844      0.0000         0.0000          0.3730            0.1483
  foot_accel_rms_mean 122                    -0.5040                   -0.0420                     0.4620         0.0851                         -3.9990      0.0000         0.0000          0.8661            0.1703
cadence_steps_per_min 122                     0.0731                    1.9959                     1.9228         0.2813                        

In [3]:
availability = participant.groupby(['age_group', 'label']).size().rename('participants').reset_index()
print('Age overlap availability:'); print(availability.to_string(index=False))

pred = pd.read_csv(PROCESSED / 'repeated_pooled_outer_predictions.csv')
pred = pred[pred.dataset_id.eq('voisard_2025')].copy(); pred['subject'] = pred.participant_key.str.split(':').str[-1]; pred['age'] = pred.subject.map(age_map)
pred['age_group'] = pd.cut(pred.age, bins=[17, 39, 59, 100], labels=['young_18_39', 'middle_40_59', 'older_60_plus'])
age_rows = []
for (seed, fold, age_group), frame in pred.groupby(['seed', 'fold', 'age_group'], observed=True):
    if frame.label_binary.nunique() < 2: continue
    y = frame.label_binary.to_numpy()
    for kind, probability, threshold in [('raw', frame.raw_probability.to_numpy(), 0.5), ('calibrated', frame.calibrated_probability.to_numpy(), frame.threshold.iloc[0])]:
        age_rows.append({'seed': seed, 'fold': fold, 'age_group': str(age_group), 'probability_type': kind, 'participants': len(frame), 'stroke_n': int(y.sum()), 'healthy_n': int((1-y).sum()), 'roc_auc': stats.mstats.hdquantiles(probability)[0] if False else __import__('sklearn.metrics').metrics.roc_auc_score(y, probability), 'balanced_accuracy': __import__('sklearn.metrics').metrics.balanced_accuracy_score(y, (probability >= threshold).astype(int)), 'threshold': threshold})
age_stratified = pd.DataFrame(age_rows)
print('Age-overlap gait model results:'); print(age_stratified.groupby(['age_group','probability_type'])[['roc_auc','balanced_accuracy','participants']].agg(['mean','std']).round(3).to_string())

Age overlap availability:
    age_group label  participants
  young_18_39   CVA             0
  young_18_39    HS            43
 middle_40_59   CVA            27
 middle_40_59    HS            15
older_60_plus   CVA            22
older_60_plus    HS            15
Age-overlap gait model results:
                               roc_auc        balanced_accuracy        participants       
                                  mean    std              mean    std         mean    std
age_group     probability_type                                                            
middle_40_59  calibrated         0.953  0.129             0.876  0.173          8.4  2.667
              raw                0.953  0.129             0.891  0.146          8.4  2.667
older_60_plus calibrated         0.913  0.107             0.777  0.161          7.4  1.242
              raw                0.913  0.107             0.816  0.160          7.4  1.242


C:\Users\frank\AppData\Local\Temp\ipykernel_28300\4232181429.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  availability = participant.groupby(['age_group', 'label']).size().rename('participants').reset_index()


In [4]:
interactions.to_csv(PROCESSED / 'age_group_feature_interactions.csv', index=False)
availability.to_csv(PROCESSED / 'age_group_label_availability.csv', index=False)
age_stratified.to_csv(PROCESSED / 'age_stratified_gait_results.csv', index=False)
print('Saved age interaction and age-stratified gait outputs.')

Saved age interaction and age-stratified gait outputs.


## Interpretation gate

A significant interaction means the age slope differs between healthy and stroke participants; it does not automatically mean age should be used as a classifier input. Age-stratified gait performance is only estimable where both labels are present. The youngest Voisard band contains healthy participants only, so it cannot support a within-age stroke-versus-healthy performance estimate.